# PokéAtlas — Preparación de la base de conocimiento

Este notebook prepara las fuentes que utilizará **PokéAtlas**, un asistente experto en información de Pokémon.

El objetivo de esta fase es generar cuatro fuentes limpias y reproducibles:

1. `pokemon_complete_clean.csv` — especies/formas, estadísticas, tipos y datos generales.
2. `type_relations.csv` — efectividad entre los 18 tipos estándar.
3. `moves.csv` — movimientos, tipo, potencia, precisión, PP, categoría y efecto.
4. `abilities.csv` — habilidades utilizadas por los Pokémon de nuestro dataset y su descripción.

Los datos adicionales se obtienen de los CSV oficiales que alimentan **PokéAPI**.

> Este notebook todavía no utiliza Gemini, ChromaDB ni LangGraph. Su única función es preparar la base documental de PokéAtlas.


## 0. Instalación

Si falta alguna dependencia, ejecutar una vez:

```bash
pip install pandas requests
```


In [1]:
from pathlib import Path
import pandas as pd
import requests
from io import StringIO

DATA_RAW = Path("../data/raw")
DATA_KNOWLEDGE = Path("../data/knowledge")

DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_KNOWLEDGE.mkdir(parents=True, exist_ok=True)

POKEMON_PATH = DATA_RAW / "pokemon_complete.csv"

if not POKEMON_PATH.exists():
    raise FileNotFoundError(
        f"No se encuentra {POKEMON_PATH}. "
        "Coloca pokemon_complete.csv dentro de data/raw/."
    )

pokemon = pd.read_csv(POKEMON_PATH)

print("pokemon_complete.csv:", pokemon.shape)
display(pokemon.head())


pokemon_complete.csv: (1350, 31)


,pokedex_number,name,type_1,type_2,hp,attack,defense,sp_attack,sp_defense,speed,...,shape,egg_groups,habitat,growth_rate,capture_rate,base_happiness,genus,evolution_chain_id,flavor_text,sprite_url
0,1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,...,quadruped,monster|plant,grassland,medium-slow,45,70,Seed PokÃ©mon,1,A strange seed was planted on its back at birt...,https://raw.githubusercontent.com/PokeAPI/spri...
1,2,Ivysaur,Grass,Poison,60,62,63,80,80,60,...,quadruped,monster|plant,grassland,medium-slow,45,70,Seed PokÃ©mon,1,"When the bulb on its back grows large, it appe...",https://raw.githubusercontent.com/PokeAPI/spri...
2,3,Venusaur,Grass,Poison,80,82,83,100,100,80,...,quadruped,monster|plant,grassland,medium-slow,45,70,Seed PokÃ©mon,1,The plant blooms when it is absorbing solar en...,https://raw.githubusercontent.com/PokeAPI/spri...
3,4,Charmander,Fire,NaN,39,52,43,60,50,65,...,upright,monster|dragon,mountain,medium-slow,45,70,Lizard PokÃ©mon,2,"Obviously prefers hot places. When it rains, s...",https://raw.githubusercontent.com/PokeAPI/spri...
4,5,Charmeleon,Fire,NaN,58,64,58,80,65,80,...,upright,monster|dragon,mountain,medium-slow,45,70,Flame PokÃ©mon,2,"When it swings its burning tail, it elevates t...",https://raw.githubusercontent.com/PokeAPI/spri...


## 1. Limpiar `pokemon_complete.csv`

El CSV original contiene algunos textos con un problema de codificación, por ejemplo `PokÃ©mon` en lugar de `Pokémon`.

Conservamos el archivo original intacto y generamos una copia limpia.


In [2]:
def fix_mojibake(value):
    if not isinstance(value, str):
        return value

    if not any(mark in value for mark in ("Ã", "Â", "â")):
        return value

    try:
        repaired = value.encode("latin1").decode("utf-8")

        before = sum(value.count(mark) for mark in ("Ã", "Â", "â"))
        after = sum(repaired.count(mark) for mark in ("Ã", "Â", "â"))

        return repaired if after < before else value

    except (UnicodeEncodeError, UnicodeDecodeError):
        return value


pokemon_clean = pokemon.copy()

for col in pokemon_clean.select_dtypes(include="object").columns:
    pokemon_clean[col] = pokemon_clean[col].map(fix_mojibake)

pokemon_clean_path = DATA_KNOWLEDGE / "pokemon_complete_clean.csv"
pokemon_clean.to_csv(pokemon_clean_path, index=False, encoding="utf-8")

print("Archivo guardado:", pokemon_clean_path)
print("Dimensiones:", pokemon_clean.shape)

# Comprobación rápida
display(
    pokemon_clean[
        ["name", "type_1", "type_2", "genus", "flavor_text"]
    ].head()
)


Archivo guardado: ..\data\knowledge\pokemon_complete_clean.csv
Dimensiones: (1350, 31)


,name,type_1,type_2,genus,flavor_text
0,Bulbasaur,Grass,Poison,Seed Pokémon,A strange seed was planted on its back at birt...
1,Ivysaur,Grass,Poison,Seed Pokémon,"When the bulb on its back grows large, it appe..."
2,Venusaur,Grass,Poison,Seed Pokémon,The plant blooms when it is absorbing solar en...
3,Charmander,Fire,NaN,Lizard Pokémon,"Obviously prefers hot places. When it rains, s..."
4,Charmeleon,Fire,NaN,Flame Pokémon,"When it swings its burning tail, it elevates t..."


## 2. Función para descargar CSV oficiales de PokéAPI

En lugar de hacer cientos de peticiones individuales a la API REST, utilizamos los **CSV oficiales del repositorio de PokéAPI**.

Esto hace el proceso más rápido y reproducible.


In [3]:
POKEAPI_CSV_BASE = (
    "https://raw.githubusercontent.com/"
    "PokeAPI/pokeapi/master/data/v2/csv"
)

def load_pokeapi_csv(filename, timeout=30):
    url = f"{POKEAPI_CSV_BASE}/{filename}"

    response = requests.get(url, timeout=timeout)
    response.raise_for_status()

    df = pd.read_csv(StringIO(response.text))

    print(f"{filename}: {df.shape}")
    return df


## 3. Relaciones entre tipos

PokéAtlas trabajará con los **18 tipos estándar** presentes en nuestro dataset.

La tabla se construye a partir de `types.csv` y `type_efficacy.csv`. El factor de daño de PokéAPI utiliza:

- `200` → ×2
- `100` → ×1
- `50` → ×0.5
- `0` → inmunidad


In [4]:
types_raw = load_pokeapi_csv("types.csv")
efficacy_raw = load_pokeapi_csv("type_efficacy.csv")

dataset_types = sorted(
    set(pokemon_clean["type_1"].dropna().str.lower())
    | set(pokemon_clean["type_2"].dropna().str.lower())
)

print("\nTipos presentes en pokemon_complete:")
print(dataset_types)
print("Número de tipos:", len(dataset_types))

type_map = (
    types_raw[types_raw["identifier"].isin(dataset_types)]
    [["id", "identifier"]]
    .set_index("id")["identifier"]
    .to_dict()
)

eff = efficacy_raw[
    efficacy_raw["damage_type_id"].isin(type_map)
    & efficacy_raw["target_type_id"].isin(type_map)
].copy()

eff["attacking_type"] = eff["damage_type_id"].map(type_map)
eff["defending_type"] = eff["target_type_id"].map(type_map)

relations = []

for type_name in dataset_types:
    outgoing = eff[eff["attacking_type"] == type_name]
    incoming = eff[eff["defending_type"] == type_name]

    def names_for(frame, factor, column):
        return "|".join(
            sorted(frame.loc[frame["damage_factor"] == factor, column].tolist())
        )

    relations.append({
        "type": type_name,
        "double_damage_to": names_for(outgoing, 200, "defending_type"),
        "half_damage_to": names_for(outgoing, 50, "defending_type"),
        "no_damage_to": names_for(outgoing, 0, "defending_type"),
        "double_damage_from": names_for(incoming, 200, "attacking_type"),
        "half_damage_from": names_for(incoming, 50, "attacking_type"),
        "no_damage_from": names_for(incoming, 0, "attacking_type"),
    })

type_relations = pd.DataFrame(relations)

type_relations_path = DATA_KNOWLEDGE / "type_relations.csv"
type_relations.to_csv(type_relations_path, index=False, encoding="utf-8")

print("\nArchivo guardado:", type_relations_path)
print("Dimensiones:", type_relations.shape)
display(type_relations.head(18))


types.csv: (21, 4)
type_efficacy.csv: (324, 3)

Tipos presentes en pokemon_complete:
['bug', 'dark', 'dragon', 'electric', 'fairy', 'fighting', 'fire', 'flying', 'ghost', 'grass', 'ground', 'ice', 'normal', 'poison', 'psychic', 'rock', 'steel', 'water']
Número de tipos: 18

Archivo guardado: ..\data\knowledge\type_relations.csv
Dimensiones: (18, 7)


,type,double_damage_to,half_damage_to,no_damage_to,double_damage_from,half_damage_from,no_damage_from
0,bug,dark|grass|psychic,fairy|fighting|fire|flying|ghost|poison|steel,,fire|flying|rock,fighting|grass|ground,
1,dark,ghost|psychic,dark|fairy|fighting,,bug|fairy|fighting,dark|ghost,psychic
2,dragon,dragon,steel,fairy,dragon|fairy|ice,electric|fire|grass|water,
3,electric,flying|water,dragon|electric|grass,ground,ground,electric|flying|steel,
4,fairy,dark|dragon|fighting,fire|poison|steel,,poison|steel,bug|dark|fighting,dragon
5,fighting,dark|ice|normal|rock|steel,bug|fairy|flying|poison|psychic,ghost,fairy|flying|psychic,bug|dark|rock,
6,fire,bug|grass|ice|steel,dragon|fire|rock|water,,ground|rock|water,bug|fairy|fire|grass|ice|steel,
7,flying,bug|fighting|grass,electric|rock|steel,,electric|ice|rock,bug|fighting|grass,ground
8,ghost,ghost|psychic,dark,normal,dark|ghost,bug|poison,fighting|normal
9,grass,ground|rock|water,bug|dragon|fire|flying|grass|poison|steel,,bug|fire|flying|ice|poison,electric|grass|ground|water,


## 4. Movimientos

Construimos una tabla compacta con la información más útil para PokéAtlas:

- nombre;
- generación;
- tipo;
- categoría (`physical`, `special` o `status`);
- potencia;
- precisión;
- PP;
- prioridad;
- efecto breve.

Los movimientos sin potencia o precisión numérica no son necesariamente errores: muchos movimientos de estado o de daño variable utilizan valores nulos.


In [5]:
moves_raw = load_pokeapi_csv("moves.csv")
move_effects_raw = load_pokeapi_csv("move_effect_prose.csv")

# Mapa de tipos
types_lookup = types_raw.set_index("id")["identifier"].to_dict()

# Clases de daño estables en PokéAPI:
damage_class_map = {
    1: "status",
    2: "physical",
    3: "special",
}

# Efectos en inglés (language_id 9)
move_effects_en = (
    move_effects_raw[move_effects_raw["local_language_id"] == 9]
    [["move_effect_id", "short_effect"]]
    .drop_duplicates(subset=["move_effect_id"])
)

moves = moves_raw.copy()

moves["type"] = moves["type_id"].map(types_lookup)
moves["damage_class"] = moves["damage_class_id"].map(damage_class_map)

moves = moves.merge(
    move_effects_en,
    left_on="effect_id",
    right_on="move_effect_id",
    how="left",
)

generation_map = {
    1: "gen-i",
    2: "gen-ii",
    3: "gen-iii",
    4: "gen-iv",
    5: "gen-v",
    6: "gen-vi",
    7: "gen-vii",
    8: "gen-viii",
    9: "gen-ix",
}

moves["generation"] = moves["generation_id"].map(generation_map)

def insert_effect_chance(row):
    text = row.get("short_effect")
    chance = row.get("effect_chance")

    if not isinstance(text, str):
        return text

    if "$effect_chance" in text and pd.notna(chance):
        return text.replace("$effect_chance", str(int(chance)))

    return text

moves["short_effect"] = moves.apply(insert_effect_chance, axis=1)

moves = moves[
    [
        "id",
        "identifier",
        "generation",
        "type",
        "damage_class",
        "power",
        "accuracy",
        "pp",
        "priority",
        "effect_chance",
        "short_effect",
    ]
].rename(columns={"identifier": "move"})

moves_path = DATA_KNOWLEDGE / "moves.csv"
moves.to_csv(moves_path, index=False, encoding="utf-8")

print("\nArchivo guardado:", moves_path)
print("Dimensiones:", moves.shape)

# Ejemplo útil para la futura demo
display(
    moves[moves["move"].isin(["earthquake", "outrage", "surf", "thunderbolt"])]
)


moves.csv: (937, 15)
move_effect_prose.csv: (872, 4)

Archivo guardado: ..\data\knowledge\moves.csv
Dimensiones: (937, 11)


,id,move,generation,type,damage_class,power,accuracy,pp,priority,effect_chance,short_effect
56,57,surf,gen-i,water,special,90.0,100.0,15.0,0,NaN,Inflicts regular damage and can hit Dive users.
84,85,thunderbolt,gen-i,electric,special,90.0,100.0,15.0,0,10.0,Has a chance to paralyze the target.
88,89,earthquake,gen-i,ground,physical,100.0,100.0,10.0,0,NaN,Inflicts regular damage and can hit Dig users.
199,200,outrage,gen-ii,dragon,physical,120.0,100.0,10.0,0,NaN,"Hits every turn for 2-3 turns, then confuses t..."


## 5. Habilidades

Nuestro `pokemon_complete.csv` ya indica qué habilidades puede tener cada Pokémon. Para no añadir información innecesaria, descargamos la descripción oficial y conservamos **solo las habilidades que aparecen realmente en nuestro dataset**.


In [6]:
abilities_raw = load_pokeapi_csv("abilities.csv")
ability_prose_raw = load_pokeapi_csv("ability_prose.csv")

# Extraer habilidades presentes en pokemon_complete
ability_names = set()

for value in pokemon_clean["abilities"].dropna():
    ability_names.update(
        part.strip().lower()
        for part in str(value).split("|")
        if part.strip()
    )

ability_names.update(
    pokemon_clean["hidden_ability"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .tolist()
)

ability_prose_en = (
    ability_prose_raw[ability_prose_raw["local_language_id"] == 9]
    [["ability_id", "short_effect", "effect"]]
    .drop_duplicates(subset=["ability_id"])
)

abilities = abilities_raw[
    abilities_raw["identifier"].isin(ability_names)
].copy()

abilities = abilities.merge(
    ability_prose_en,
    left_on="id",
    right_on="ability_id",
    how="left",
)

abilities["generation"] = abilities["generation_id"].map(generation_map)

abilities = abilities[
    [
        "id",
        "identifier",
        "generation",
        "is_main_series",
        "short_effect",
        "effect",
    ]
].rename(columns={"identifier": "ability"})

abilities_path = DATA_KNOWLEDGE / "abilities.csv"
abilities.to_csv(abilities_path, index=False, encoding="utf-8")

print("\nArchivo guardado:", abilities_path)
print("Dimensiones:", abilities.shape)
print("Habilidades únicas detectadas en pokemon_complete:", len(ability_names))

display(
    abilities[
        abilities["ability"].isin(
            ["rough-skin", "torrent", "intimidate", "multiscale"]
        )
    ]
)


abilities.csv: (374, 4)
ability_prose.csv: (810, 4)

Archivo guardado: ..\data\knowledge\abilities.csv
Dimensiones: (310, 6)
Habilidades únicas detectadas en pokemon_complete: 310


,id,ability,generation,is_main_series,short_effect,effect
21,22,intimidate,gen-iii,1,Lowers opponents' Attack one stage upon enteri...,"When this Pokémon enters battle, the opponent'..."
23,24,rough-skin,gen-iii,1,Damages attacking Pokémon for 1/8 their max HP...,Whenever a move makes contact with this Pokémo...
66,67,torrent,gen-iii,1,Strengthens Water moves to inflict 1.5× damage...,When this Pokémon has 1/3 or less of its HP re...
135,136,multiscale,gen-v,1,Halves damage taken from full HP.,This Pokémon takes half as much damage when it...


## 6. Validaciones finales

Comprobamos que las cuatro fuentes existen y que tienen contenido.

También mostramos algunas filas que utilizaremos más adelante para probar el RAG de PokéAtlas.


In [7]:
files = {
    "pokemon": pokemon_clean_path,
    "types": type_relations_path,
    "moves": moves_path,
    "abilities": abilities_path,
}

print("FUENTES GENERADAS")
print("=" * 60)

for name, path in files.items():
    df_check = pd.read_csv(path)
    print(f"{name:10s} -> {path.name:28s} {df_check.shape}")

assert len(type_relations) == 18, "Esperábamos exactamente 18 tipos."
assert len(moves) > 500, "La tabla de movimientos parece incompleta."
assert len(abilities) > 100, "La tabla de habilidades parece incompleta."

print("\n✅ Validaciones básicas superadas.")


FUENTES GENERADAS
pokemon    -> pokemon_complete_clean.csv   (1350, 31)
types      -> type_relations.csv           (18, 7)
moves      -> moves.csv                    (937, 11)
abilities  -> abilities.csv                (310, 6)

✅ Validaciones básicas superadas.


In [8]:
# Pokémon para futuras pruebas de PokéAtlas
display(
    pokemon_clean[
        pokemon_clean["name"].str.lower().isin(
            ["garchomp", "dragonite", "blastoise", "salamence", "tyranitar"]
        )
    ][
        [
            "name",
            "type_1",
            "type_2",
            "hp",
            "attack",
            "defense",
            "sp_attack",
            "sp_defense",
            "speed",
            "abilities",
            "hidden_ability",
            "generation",
        ]
    ]
)


,name,type_1,type_2,hp,attack,defense,sp_attack,sp_defense,speed,abilities,hidden_ability,generation
8,Blastoise,Water,NaN,79,83,100,85,105,78,torrent|rain-dish,rain-dish,gen-i
148,Dragonite,Dragon,Flying,91,134,95,100,100,80,inner-focus|multiscale,multiscale,gen-i
247,Tyranitar,Rock,Dark,100,134,110,95,100,61,sand-stream|unnerve,unnerve,gen-ii
372,Salamence,Dragon,Flying,95,135,80,110,80,100,intimidate|moxie,moxie,gen-iii
444,Garchomp,Dragon,Ground,108,130,95,80,85,102,sand-veil|rough-skin,rough-skin,gen-iv


## Resultado

Después de ejecutar el notebook tendremos:

```text
data/
├── raw/
│   └── pokemon_complete.csv
└── knowledge/
    ├── pokemon_complete_clean.csv
    ├── type_relations.csv
    ├── moves.csv
    └── abilities.csv
```

Estas cuatro tablas formarán la **base documental inicial de PokéAtlas**.

El siguiente notebook transformará sus filas en documentos de texto, aplicará chunking, generará embeddings con Gemini y los indexará en ChromaDB.
